# Code-intelligence source availability amendment

**Status:** approach approved; authoritative design awaiting final user review  
**Date:** 2026-08-29  
**Design epic:** `bd-1fud`  
**Parent live-run epic:** `bd-11qh`  
**Amends:** `2026-08-27-code-intelligence-industry-benchmark-design.ipynb`  
**Approved approach:** availability-aware immutable source locking and denominator accounting

This amendment makes a real, truthful run possible when upstream benchmark repositories or revisions have disappeared. It never fabricates source, silently drops cases, or calls an available-corpus score a complete-source benchmark.

Formal profile pin: `relational_lia` version 1, Z3 `qf_lia_bool_int_enum`. Native `ns_mermaid` cells are the authoritative policy relations. The Notebook MCP transport was unavailable (`Transport closed`) during authoring; the infrastructure exception is recorded on `bd-1fud`. The equivalent catalog and generic-solver obligations were executed and are listed in the evidence section.

## 1. Evidence and decision

The pinned CrossCodeEval archive contains 1,002 unique repository identities. The exhaustive census found:

- 945 exact objects at their original source;
- 5 exact, provenance-qualified archival recoveries;
- 52 identities still unavailable after original-host, mirror, and Software Heritage investigation;
- zero authentication failures and zero original-source transient/rate failures.

The strict complete-lock workflow is therefore unsatisfiable today (`sol_db797cfdd5de48bb`). Waiting indefinitely would produce no score. Relaxing only a smoke filter would not solve the complete run.

The accepted decision is to add a source-availability dimension orthogonal to dataset eligibility. Every repository identity and every dataset case remains denominator-visible. Verified snapshots may run; unavailable snapshots terminate with typed evidence. Any unavailable eligible case prevents a complete-source or publishable claim.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-SOURCE-STATE
@type SourceState = enum[resolved_original, resolved_archive, source_unavailable]
@input original_available: Bool
@input archive_authoritative: Bool
@output status: SourceState
@requires INPUT_DOMAIN: true`"]

    ORIGINAL["`@branch ORIGINAL
@when original_available
@ensures ORIGINAL_STATUS: status = resolved_original`"]

    ARCHIVE["`@branch ARCHIVE
@when not original_available and archive_authoritative
@ensures ARCHIVE_STATUS: status = resolved_archive`"]

    UNAVAILABLE["`@branch UNAVAILABLE
@when not original_available and not archive_authoritative
@ensures UNAVAILABLE_STATUS: status = source_unavailable`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATES: witness each status`"]

    SPEC --> ORIGINAL --> CHECK
    SPEC --> ARCHIVE --> CHECK
    SPEC --> UNAVAILABLE --> CHECK

## 2. Source lock v2

Bump the canonical schema to `spur-live-source-lock-v2`. A lock contains one immutable row per required repository identity, sorted by canonical `(repository_uri, requested_revision, subdirectory)`. The lock as a whole binds the dataset hashes, adapter/scorer versions, census evidence manifest digest, and canonical policy hash.

All rows retain the requested original URI and revision, license metadata, a typed state, and an evidence digest. State-specific fields are closed:

| State | Required snapshot fields | Required provenance | Forbidden fields |
|---|---|---|---|
| `resolved_original` | full commit, subdirectory, materialization hash | original URI/revision and verification evidence | archival URI |
| `resolved_archive` | full commit, subdirectory, materialization hash | original identity, archival URI, authoritative relationship, verification evidence | none of the required snapshot fields |
| `source_unavailable` | none | frozen diagnostic, attempted authorities, evidence digest | full commit, materialization hash, archival URI |

A recovery is authoritative only when evidence binds the recovered Git object to the requested original identity and revision. Repository-name similarity, prompt reconstruction, arbitrary forks, and unverified mirrors are rejected. Observation time may be recorded as evidence metadata but does not replace content-addressed identity.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-LOCK-SHAPE
@type SourceState = enum[resolved_original, resolved_archive, source_unavailable]
@type LockValidity = enum[reject, accept]
@input state: SourceState
@input full_commit_present: Bool
@input materialization_hash_present: Bool
@input archive_uri_present: Bool
@input evidence_digest_present: Bool
@output status: LockValidity
@requires INPUT_DOMAIN: true`"]

    REJECT["`@branch REJECT
@when not evidence_digest_present or (state = resolved_original and not (full_commit_present and materialization_hash_present and not archive_uri_present)) or (state = resolved_archive and not (full_commit_present and materialization_hash_present and archive_uri_present)) or (state = source_unavailable and (full_commit_present or materialization_hash_present or archive_uri_present))
@ensures REJECT_STATUS: status = reject`"]

    ACCEPT["`@branch ACCEPT
@when evidence_digest_present and ((state = resolved_original and full_commit_present and materialization_hash_present and not archive_uri_present) or (state = resolved_archive and full_commit_present and materialization_hash_present and archive_uri_present) or (state = source_unavailable and not full_commit_present and not materialization_hash_present and not archive_uri_present))
@ensures ACCEPT_STATUS: status = accept`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]

    SPEC --> REJECT --> CHECK
    SPEC --> ACCEPT --> CHECK

## 3. Materialization and execution

Materialization accepts only `resolved_original` and `resolved_archive` rows. It verifies the complete Git object, checks the locked subdirectory, recomputes the normalized materialization hash, and refuses mutable or mismatched content. `source_unavailable` is terminal and never reaches Git checkout, indexing, retrieval, or model execution.

Source availability does not change upstream eligibility. A supported case whose required repository row is unavailable remains an eligible case with terminal outcome `source_unavailable`; it is not recast as unsupported or invalid. Unsupported and malformed cases retain their existing adapter semantics.

Checkpoint identity includes source-lock schema/version, row state and evidence digest, repository materialization hash when present, case input hash, evaluator version, policy hash, and backend identity. A state or evidence change invalidates resume and cache hits.

## 4. Denominator conservation and run scope

For each suite, eligible cases must conserve exactly:

`eligible_total = evaluated + source_unavailable + filtered + execution_failed`

The buckets are disjoint terminal outcomes. `evaluated` means a verified checkpoint reached the suite scorer. `source_unavailable` means no verified immutable snapshot exists. `filtered` means the operator explicitly selected a subset. `execution_failed` means a resolved case failed before a verified score.

The report carries two denominator views without ambiguity:

- manifest coverage: eligible total and every terminal bucket;
- evaluated metrics: native suite numerators and denominators over verified evaluated checkpoints only.

A run with no filter or execution failure but at least one unavailable source is `available_corpus`. It is a real benchmark result over the verifiably materializable corpus, but not a complete-source result. A filtered run is `filtered_partial`. Only zero unavailable, zero filtered, zero failed, and full evaluated coverage is `complete_source`.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-RUN-SCOPE
@type RunScope = enum[invalid_accounting, execution_incomplete, filtered_partial, available_corpus, complete_source]
@input eligible_total: Int
@input evaluated: Int
@input source_unavailable: Int
@input filtered: Int
@input execution_failed: Int
@output status: RunScope
@requires NONNEGATIVE: eligible_total >= 0 and evaluated >= 0 and source_unavailable >= 0 and filtered >= 0 and execution_failed >= 0`"]

    INVALID["`@branch INVALID
@when eligible_total != evaluated + source_unavailable + filtered + execution_failed
@ensures INVALID_STATUS: status = invalid_accounting`"]
    FAILED["`@branch FAILED
@when eligible_total = evaluated + source_unavailable + filtered + execution_failed and execution_failed > 0
@ensures FAILED_STATUS: status = execution_incomplete`"]
    FILTERED["`@branch FILTERED
@when eligible_total = evaluated + source_unavailable + filtered + execution_failed and execution_failed = 0 and filtered > 0
@ensures FILTERED_STATUS: status = filtered_partial`"]
    AVAILABLE["`@branch AVAILABLE
@when eligible_total = evaluated + source_unavailable + filtered + execution_failed and execution_failed = 0 and filtered = 0 and source_unavailable > 0
@ensures AVAILABLE_STATUS: status = available_corpus`"]
    COMPLETE["`@branch COMPLETE
@when eligible_total = evaluated + source_unavailable + filtered + execution_failed and execution_failed = 0 and filtered = 0 and source_unavailable = 0
@ensures COMPLETE_STATUS: status = complete_source`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify SCOPES: witness each status`"]
    SPEC --> INVALID --> CHECK
    SPEC --> FAILED --> CHECK
    SPEC --> FILTERED --> CHECK
    SPEC --> AVAILABLE --> CHECK
    SPEC --> COMPLETE --> CHECK

## 5. Scoring, reporting, and publication

RepoQA, CrossCodeEval, and JCG native metrics are recomputed only from verified `evaluated` checkpoints. Each metric record includes `eligible_total`, `evaluated`, `source_unavailable`, `filtered`, `execution_failed`, answer rate over the manifest, and the evaluated metric denominator. No unavailable case is assigned zero-quality retrieval evidence because no retrieval occurred; it is counted separately.

Every report includes the complete lock digest, census evidence digest, source-state counts by suite/repository/language, unavailable identity list and reasons, revision/dirty state, argv, timings, peak RSS, storage, policy hash, scorer/adapter versions, and checkpoint verification summary. JSON and checksum behavior remain deterministic.

An `available_corpus` report may be generated and inspected. It must prominently state that it is not directly comparable to a complete-source result unless both reports use the same lock and availability set. Publication remains fail-closed: only `complete_source` with verified lock, complete metrics, and passing deterministic gates may receive `publish_deterministic`.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-REPORT-GATE
@type ReleaseStatus = enum[reject, publish_deterministic]
@input accounting_complete: Bool
@input source_unavailable_present: Bool
@input filtered_present: Bool
@input execution_failed_present: Bool
@input lock_verified: Bool
@input metrics_complete: Bool
@input deterministic_pass: Bool
@output status: ReleaseStatus
@requires INPUT_DOMAIN: true`"]

    REJECT["`@branch REJECT
@when not accounting_complete or source_unavailable_present or filtered_present or execution_failed_present or not lock_verified or not metrics_complete or not deterministic_pass
@ensures REJECT_STATUS: status = reject`"]
    PUBLISH["`@branch PUBLISH
@when accounting_complete and not source_unavailable_present and not filtered_present and not execution_failed_present and lock_verified and metrics_complete and deterministic_pass
@ensures PUBLISH_STATUS: status = publish_deterministic`"]

    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> REJECT --> CHECK
    SPEC --> PUBLISH --> CHECK

## 6. CLI, migration, and compatibility

The public workflow remains `validate → index → retrieve → score → report → publish`. `validate` creates or verifies lock v2 and emits the availability census. `index` and `retrieve` skip only typed unavailable rows while writing terminal per-case checkpoints. `score` verifies conservation before computing metrics. `report` always writes a valid complete accounting report; `publish` enforces the formal gate.

Add machine-readable selectors for run scope and terminal outcomes, but do not add a flag that silently weakens the gate. A user may explicitly filter suites/cases; that remains `filtered_partial`.

Read existing v1 resolved-only locks and deterministically upgrade them to v2 resolved rows after verification. Write only v2. Old checkpoints are invalidated because the lock identity and accounting schema change. Preserve existing fixture reports through an explicit fixture contract version; do not reinterpret them as live available-corpus reports.

## 7. Security and failure handling

- Continue rejecting traversal, links, special archive entries, mixed roots, revision conflicts, dirty materializations, and checksum mismatch.
- Never execute repository code during recovery or validation.
- Treat an archival relationship as data requiring evidence, not as a trusted hostname heuristic.
- Canonicalize diagnostics so secrets, local paths, and mutable transport details do not enter deterministic report identity.
- Preserve full raw attempt logs in the evidence directory with a verified manifest, while the canonical lock stores only stable diagnostics and evidence digests.
- A newly available source requires a new lock digest and invalidates dependent checkpoints; it does not mutate an existing published lock.
- Any state/field inconsistency, denominator mismatch, duplicate identity, unverifiable archive, or checkpoint tamper fails closed.

## 8. Verification and acceptance

Implementation follows RED/GREEN TDD and must include:

1. lock-v2 serialization, canonical hashing, v1 migration, duplicate detection, and all state/field combinations;
2. positive archival provenance fixtures plus name-only mirror, unrelated-prefix, and tampered-evidence rejection;
3. unavailable rows never invoking materialization/index/backend calls;
4. case conservation across RepoQA, CrossCodeEval, and JCG, including one repository shared by multiple cases;
5. resumed-checkpoint invalidation for state, evidence, policy, and schema changes;
6. native metric recomputation from evaluated checkpoints only, with manifest/evaluated denominators both present;
7. available-corpus report generation and publish rejection;
8. complete-source publication regression;
9. real pinned-corpus validate/index/retrieve/score/report run with checksums and no fabricated result;
10. `scripts/spur-cargo test -p spur-code-eval`, crate Clippy with `-D warnings`, rustdoc, CLI help, and existing semantic benchmark.

Acceptance requires a truthful nonzero real evaluated denominator for each runnable selected suite, exact accounting for every eligible case, deterministic rerun/resume, and explicit `reject` whenever any eligible case is source-unavailable.

## 9. Solver and census evidence

| Obligation | Result | Evidence |
|---|---|---|
| Strict complete lock under current public availability | `unsat` | `sol_db797cfdd5de48bb` |
| Source-state partition has no missing or overlapping state | counterexample `unsat` | `sol_6c4b5df1eb5c4149` |
| Current identity census conserves `945 + 5 + 52 = 1002` | `sat` | `sol_09f236798b4d40d7` |
| Three lock row shapes satisfy the catalog `mutually_consistent` relation | `pass` / `sat` | `sol_921c8569d1b9467d` |
| Tampered unavailable row carrying a commit is rejected | `fail` / `unsat` | `sol_7594eb89d5df4a3d` |
| Available-corpus report with unavailable cases and rejected publication exists | `sat` | `sol_d90caefea6984bf5` |
| Publishing while an unavailable eligible case exists | counterexample `unsat` | `sol_61b72e31b224403f` |

Catalog rules used: `data_integrity.mutually_consistent` v1; `data_integrity.conditional_required` v1 was reviewed for field-presence semantics. Solver: Z3 4.16.0. Census checksums are rooted at `.spur/bench-evidence/bd-11qh-live/census/CENSUS-SHA256SUMS-v1` (SHA-256 `26774f8fe6d78474cb7d728a3479db6c104f0de264a3d5fa342edc3cbe4ca7a1`).

The four native NS-Mermaid cells must be preflighted and freshly executed through Notebook MCP before implementation approval once the transport is restored. Until then, their empty outputs are intentional and must not be described as fresh notebook proofs.

## 10. Planning boundary

After final design approval, create a beads-backed implementation DAG with six intent-isolated tasks:

1. source-lock v2 types, canonical validation, and migration;
2. resolver/census archival provenance and unavailable-state construction;
3. materializer/evaluator terminal unavailable handling and checkpoint identity;
4. case accounting, suite scoring denominators, report schema, and publication gate;
5. CLI/runner orchestration, resume/cache behavior, and integration tests;
6. real available-corpus benchmark execution, checksum verification, regression/Clippy/rustdoc gates, and evidence handoff.

The plan may split a task only when file ownership or dependency structure requires it. No implementation begins from this notebook alone; final user approval and the writing-plans workflow are required.